In [ ]:
import json
import pandas as pd
import math
# Replace 'data.json' with the path to your JSON file
# file_path = 'matillion_full.json'
# filter_file_path = 'matillion_full_filter.json'
# file_path = 'matillion_delta.json'
file_path = 'delta_filled.json'

# Open and load the JSON file
with open(file_path, 'r', encoding='utf-8') as file:
    data = json.loads(file.read())
# Open and load the JSON file
with open(file_path, 'r', encoding='utf-8') as filter_file:
    filter_data = json.loads(filter_file.read())



table_header_df = pd.read_excel('table_header - GA.xlsx', keep_default_na=False)
# table_header_df = pd.read_excel('table_header 2.xlsx')
table_detail_raw_original_df = pd.read_excel('table_detail - GA.xlsx', keep_default_na=False)
table_detail_raw_df = table_detail_raw_original_df[(table_detail_raw_original_df['target_table_name'].notna())]

def column_priority(col):
    if col.lower() == 'valid_from':
        return 999
    elif col.lower().endswith('_key') and (
            col.lower().startswith('mde_') 
            or col.lower().startswith('tde_')
            or col.lower().startswith('fact_') 
            or col.lower().startswith('dim_')
    ):
        return 0
    elif col.lower().endswith('_bk'):
        return 1
    else:
        return 2


table_detail_raw_df['bk_priority'] = table_detail_raw_df['target_column'].apply(column_priority)


table_detail_df = table_detail_raw_df.sort_values(
    by=['target_table_name', 'primary_key_flag', 'bk_priority', 'target_column'],
    ascending=[True, False, True, True]
).drop(columns='bk_priority') 

data_type_mapping = {
"TEXT(255)": "TEXT(255)",
"TIMESTAMP_NTZ": "TIMESTAMP_NTZ",
"TIMESTAMP_NTZ": "TIMESTAMP_NTZ",
"BOOLEAN": "BOOLEAN",
"NUMBER": "NUMBER",
"text": "TEXT",
"Text": "TEXT",
"Date & Time": "TIMESTAMP_NTZ",
"Decimal (38,4)": "NUMBER(38,5)",
"decimal": "NUMBER(38,5)",
"Boolean": "BOOLEAN",
"flag": "BOOLEAN",
"Integer": "NUMBER",
"integer": "NUMBER",
"Date & Time": "TIMESTAMP_NTZ",
"datetime": "TIMESTAMP_NTZ",
"Date": "DATE"
}

for index,row in table_header_df.iterrows():
    print(index)
    print(row)
    pl_table_lower = row['pl_name'].lower()
    pl_table_upper = row['pl_name'].upper()
    pl_table_historization_upper = row['historization'].upper()
    pl_schema_upper = row['pl_schema'].upper()
    staging_schema_upper = row['staging_schema']
    staging_table_upper = row['staging_table'].upper()
    pl_table_description = row['pl_description']
    pl_filter_column = row['filter_column']
    pl_filter_qualifier = row['filter_qualifier']
    pl_filter_operator = row['filter_operator']
    pl_filter_column_value = row['filter_column_value']
    
    print(pl_filter_column)
    print(pl_filter_qualifier)
    print(pl_filter_operator)
    print(pl_filter_column_value)
    if str(pl_filter_column) != '' and str(pl_filter_qualifier) != '' and str(pl_filter_operator) != '' :
    # and str(pl_filter_column_value) != 'nan':
        print('filter')
        modified_data = filter_data.copy()
    else:
        print('non filter')
        modified_data = data.copy()


    table_target_detail_df = pd.DataFrame()
    table_target_detail_df = table_detail_df[table_detail_df['target_table_name'] == pl_table_upper]
    calculator_formula_list = []
    column_description_list = []
    column_classification_list = []
    primary_key_list = []
    md5_key_list = []
    source_column_list = []
    ## prepare calculator data ##
    for index,row in table_target_detail_df.iterrows():
        target_column = row['target_column'] 
        formula = row['column_formula']
        source_column = row['source_column']
        if row['column_data_type'] not in data_type_mapping:
            print(row['column_data_type'])
            break
        else:
            column_data_type = data_type_mapping.get(row['column_data_type'])
        column_description = row['column_description']
        column_classification = row['column_classification']
        column_domain = row['column_domain']
        primary_key_flag = row['primary_key_flag']

        if pd.notna(source_column):
            source_column_list.append(source_column)
        if not pd.notna(formula):
            calculation_formula = f"{source_column}::{column_data_type}"
        else:
            calculation_formula = f"{formula}::{column_data_type}"
        if primary_key_flag == 'Y':
            primary_key_list.append({'values':[target_column]})
            md5_key_list.append([source_column])

        if target_column == 'VALID_FROM':
            if pd.notna(formula):
                md5_key_list.append([formula])
            elif pd.notna(source_column):
                md5_key_list.append([source_column])
            else:
                print(f'{target_column} has issue, no formula or source column defined')


        calculator_formula_list.append({'values':[target_column, calculation_formula]})
        if pd.notna(column_description):
            column_description_list.append({'values':[target_column, column_description.replace("'","\\'")]})


        if pd.notna(column_classification) and pd.notna(column_domain):
            column_classification_list.append({'values':[target_column, column_classification, column_domain]})
        else:
            print(f'{target_column} has issue with column_classification or/and column_domain: column_classification: {column_classification} | column_domain: {column_domain}')
        source_column_string = ','.join(source_column_list)

    # final_calculator_list = [{'values':[f'{pl_table_upper}_KEY','MD5(' + "||'|'||".join([f"UPPER(NVL({col[0]}::varchar,'null'))" for col in md5_key_list]) + "||'|'|| \"_ETL_LOAD_DATETIME\"::varchar)"]}]

    # final_calculator_list.extend(calculator_formula_list)
    gv_calculator_list = []
    for calc in calculator_formula_list:
        gv_calculator_list.append(calc)
        
        
    print(column_description_list)

    modified_data['transformationJobs'][0]['grids']['gv_calculator']['values'] = gv_calculator_list
    modified_data['orchestrationJobs'][1]['grids']['gv_primary_key_configuration']['values'] = primary_key_list
    modified_data['orchestrationJobs'][1]['grids']['gv_column_description_configuration']['values'] = column_description_list
    modified_data['orchestrationJobs'][1]['grids']['gv_column_classification_configuration']['values'] = column_classification_list


    modified_data_string = json.dumps(modified_data)


    modified_data_string = modified_data_string.replace('$$target_schema$$',pl_schema_upper)
    modified_data_string = modified_data_string.replace('$$target_table_lower$$',pl_table_lower)
    modified_data_string = modified_data_string.replace('$$target_table$$',pl_table_upper)
    modified_data_string = modified_data_string.replace('$$source_schema$$',staging_schema_upper)
    modified_data_string = modified_data_string.replace('$$source_table$$',staging_table_upper)
    modified_data_string = modified_data_string.replace('$$column_list$$',source_column_string)
    modified_data_string = modified_data_string.replace('$$table_description$$',pl_table_description.replace("'","\\\\'"))
    modified_data_string = modified_data_string.replace('$$historization$$',pl_table_historization_upper)
    if str(pl_filter_column) != 'nan' and str(pl_filter_qualifier) != 'nan' and str(pl_filter_operator) != 'nan' :
    # and str(pl_filter_column_value) != 'nan':
        modified_data_string = modified_data_string.replace('$$filter_column$$',pl_filter_column)
        modified_data_string = modified_data_string.replace('$$filter_qualifier$$',pl_filter_qualifier)
        modified_data_string = modified_data_string.replace('$$filter_operator$$',pl_filter_operator)
        modified_data_string = modified_data_string.replace('$$filter_column_value$$',pl_filter_column_value)

    # print(modified_data_string)
    with open(f'matillion_json/{pl_table_lower}.json', 'w') as jsonfile:
            print(f'matillion_json/{pl_table_lower}.json')
            jsonfile.write(modified_data_string)
 

0
staging_schema                                      STG_GOOGLE_ANALYTICS
staging_table                     T_GOOGLE_ANALYTICS_HACK_MONTHLY_MARKET
pl_schema                                                 PL_SOCIALMEDIA
pl_name                                               DIM_CUSTOM_CHANNEL
pl_description         Represents the type of website within the plat...
historization                                                SCD1_UPDATE
filter_column                                               SUB_CATEGORY
filter_qualifier                                                     NOT
filter_operator                                                     NULL
filter_column_value                                                     
Name: 0, dtype: str
SUB_CATEGORY
NOT
NULL

filter
[{'values': ['DIM_CUSTOM_CHANNEL_KEY', 'Primary Key for DIM_CUSTOM_CHANNEL']}, {'values': ['ACTIVE_FLAG', 'It indicates if the Record is Active']}, {'values': ['CUSTOM_CHANNEL_NAME', 'It shows what type of site it is w

In [2]:
import json
import pandas as pd
import math

# Replace 'data.json' with the path to your JSON file
# file_path = "matillion_full.json"
file_path = "matillion_full_filter.json"

# Open and load the JSON file
with open(file_path, "r", encoding="utf-8") as file:
    data = json.loads(file.read())


table_header_df = pd.read_excel("table_header - auto_del.xlsx")
table_detail_raw_df = pd.read_excel("table_detail - auto_del.xlsx")


def column_priority(col):
    if col.lower() == "valid_from":
        return 999
    elif col.lower().endswith("_key") and (
        col.lower().startswith("mde_")
        or col.lower().startswith("tde_")
        or col.lower().startswith("fact_")
        or col.lower().startswith("dim_")
    ):
        return 0
    elif col.lower().endswith("_bk"):
        return 1
    else:
        return 2


table_detail_raw_df["bk_priority"] = table_detail_raw_df["target_column"].apply(
    column_priority
)
display(table_detail_raw_df)


table_detail_df = table_detail_raw_df.sort_values(
    by=["target_table_name", "primary_key_flag", "bk_priority", "target_column"],
    ascending=[True, False, True, True],
).drop(columns="bk_priority")

data_type_mapping = {
    "TEXT(255)": "TEXT(255)",
    "TIMESTAMP_NTZ": "TIMESTAMP_NTZ",
    "TIMESTAMP_NTZ": "TIMESTAMP_NTZ",
    "BOOLEAN": "BOOLEAN",
    "NUMBER": "NUMBER",
    "text": "TEXT",
    "Text": "TEXT",
    "Date & Time": "TIMESTAMP_NTZ",
    "Decimal (38,4)": "NUMBER(38,5)",
    "Decimal (38,5)": "NUMBER(38,5)",
    "decimal": "NUMBER(38,5)",
    "Boolean": "BOOLEAN",
    "flag": "BOOLEAN",
    "Integer": "NUMBER",
    "integer": "NUMBER",
    "Date & Time": "TIMESTAMP_NTZ",
    "datetime": "TIMESTAMP_NTZ",
    "Date": "DATE",
    "Time": "TIMESTAMP_NTZ",
}

for index, row in table_header_df.iterrows():
    pl_table_lower = row["pl_name"].lower()
    pl_table_upper = row["pl_name"].upper()
    pl_schema_upper = row["pl_schema"].upper()
    staging_schema_upper = row["staging_schema"]
    staging_table_upper = row["staging_table"].upper()
    pl_table_description = row["pl_description"]

    modified_data = data.copy()

    table_target_detail_df = pd.DataFrame()
    table_target_detail_df = table_detail_df[
        table_detail_df["target_table_name"] == pl_table_upper
    ]
    calculator_formula_list = []
    column_description_list = []
    column_classification_list = []
    primary_key_list = []
    md5_key_list = []
    source_column_list = []
    ## prepare calculator data ##
    for index, row in table_target_detail_df.iterrows():
        target_column = row["target_column"]
        formula = row["column_formula"]
        source_column = row["source_column"]
        print(source_column)
        print(row["column_data_type"])
        if row["column_data_type"].strip() not in data_type_mapping:
            print(source_column, row["column_data_type"])
            raise Exception(
                f"{row['column_data_type']} not found in data_type_mapping!"
            )
        else:
            column_data_type = data_type_mapping.get(row["column_data_type"].strip())
            print("--")
            print(row["column_data_type"])
            print(column_data_type)
        column_description = row["column_description"]
        column_classification = row["column_classification"]
        column_domain = row["column_domain"]
        primary_key_flag = row["primary_key_flag"]

        if pd.notna(source_column):
            source_column_list.append(source_column)
        if not pd.notna(formula):
            calculation_formula = f"{source_column}::{column_data_type}"
        else:
            calculation_formula = f"{formula}::{column_data_type}"
        print(calculation_formula)
        print("----")
        if primary_key_flag == "Y":
            primary_key_list.append({"values": [target_column]})
            md5_key_list.append([source_column])

        if target_column == "VALID_FROM":
            if pd.notna(formula):
                md5_key_list.append([formula])
            elif pd.notna(source_column):
                md5_key_list.append([source_column])
            else:
                print(f"{target_column} has issue, no formula or source column defined")

        calculator_formula_list.append({"values": [target_column, calculation_formula]})
        if pd.notna(column_description):
            column_description_list.append(
                {"values": [target_column, column_description.replace("'", "\\'")]}
            )

        if pd.notna(column_classification) and pd.notna(column_domain):
            column_classification_list.append(
                {"values": [target_column, column_classification, column_domain]}
            )
        else:
            print(
                f"{target_column} has issue with column_classification or/and column_domain: column_classification: {column_classification} | column_domain: {column_domain}"
            )
        source_column_string = ",".join(source_column_list)

    # final_calculator_list = [{'values':[f'{pl_table_upper}_KEY','MD5(' + "||'|'||".join([f"UPPER(NVL({col[0]}::varchar,'null'))" for col in md5_key_list]) + "||'|'|| \"_ETL_LOAD_DATETIME\"::varchar)"]}]

    # final_calculator_list.extend(calculator_formula_list)
    gv_calculator_list = []
    for calc in calculator_formula_list:
        gv_calculator_list.append(calc)

    modified_data["transformationJobs"][0]["grids"]["gv_calculator"][
        "values"
    ] = gv_calculator_list
    modified_data["orchestrationJobs"][0]["grids"]["gv_primary_key_configuration"][
        "values"
    ] = primary_key_list
    modified_data["orchestrationJobs"][0]["grids"][
        "gv_column_description_configuration"
    ]["values"] = column_description_list
    modified_data["orchestrationJobs"][0]["grids"][
        "gv_column_classification_configuration"
    ]["values"] = column_classification_list

    modified_data_string = json.dumps(modified_data)

    modified_data_string = modified_data_string.replace(
        "$$target_schema$$", pl_schema_upper
    )
    modified_data_string = modified_data_string.replace(
        "$$target_table_lower$$", pl_table_lower
    )
    modified_data_string = modified_data_string.replace(
        "$$target_table$$", pl_table_upper
    )
    modified_data_string = modified_data_string.replace(
        "$$source_schema$$", staging_schema_upper
    )
    modified_data_string = modified_data_string.replace(
        "$$source_table$$", staging_table_upper
    )
    modified_data_string = modified_data_string.replace(
        "$$column_list$$", source_column_string
    )
    modified_data_string = modified_data_string.replace(
        "$$table_description$$", pl_table_description.replace("'", "\\\\'")
    )

    # print(modified_data_string)
    with open(f"matillion_json/{pl_table_lower}.json", "w") as jsonfile:
        print(f"matillion_json/{pl_table_lower}.json")
        jsonfile.write(modified_data_string)

,target_table_name,source_column,target_column,primary_key_flag,column_formula,column_data_type,column_description,column_classification,column_domain,bk_priority
0,TDE_AUTODELIVERY_SUBSCRIPTION,CONSUMERID,IDENTITY_UNIQUE_IDENTIFIER_BK,Y,NaN,text,Customer associated with an auto-delivery subs...,C2,CONS,1
1,TDE_AUTODELIVERY_SUBSCRIPTION,SUBSCRIPTIONID,ORDER_IDENTIFIER_BK,Y,NaN,text,Subscription instance that defines recurring d...,C3,CONS,1
2,TDE_AUTODELIVERY_SUBSCRIPTION,MARKETID,COUNTRY_CODE,NaN,NaN,text,Specific market where the auto-delivery subscr...,C3,CONS,2
3,TDE_AUTODELIVERY_SUBSCRIPTION,BASESTOREID,BASE_STORE_CODE,NaN,NaN,text,Indicator of base store from where order is ge...,C3,CONS,2
4,TDE_AUTODELIVERY_SUBSCRIPTION,FREQUENCY,DELIVERY_FREQUENCY,NaN,NaN,text,It defines how often the product or service is...,C3,CONS,2
5,TDE_AUTODELIVERY_SUBSCRIPTION,STATUS,SUBSCRIPTION_STATUS,NaN,NaN,text,Current state of the auto-delivery subscriptio...,C3,CONS,2
6,TDE_AUTODELIVERY_SUBSCRIPTION,CREATIONDATE,ORDER_CREATION_DATETIME,NaN,TRY_TO_TIMESTAMP_NTZ(CREATIONDATE),datetime,Order creation date.,C3,CONS,2
7,TDE_AUTODELIVERY_SUBSCRIPTION,STARTDATE,SUBSCRIPTION_START_DATETIME,NaN,TRY_TO_TIMESTAMP_NTZ(STARTDATE),datetime,Date when the auto-delivery subscription becom...,C3,CONS,2
8,TDE_AUTODELIVERY_SUBSCRIPTION,NEXTORDEREXECUTIONDATE,NEXT_ORDER_EXECUTION_DATETIME,NaN,TRY_TO_TIMESTAMP_NTZ(NEXTORDEREXECUTIONDATE),datetime,Scheduled date and time for the next auto-deli...,C3,CONS,2
9,TDE_AUTODELIVERY_SUBSCRIPTION,PREVIOUSORDEREXECUTIONDATE,PREVIOUS_ORDER_EXECUTION_DATETIME,NaN,TRY_TO_TIMESTAMP_NTZ(PREVIOUSORDEREXECUTIONDATE),datetime,Date and time when the last auto-delivery orde...,C3,CONS,2


CONSUMERID
text
--
text
TEXT
CONSUMERID::TEXT
----
SUBSCRIPTIONID
text
--
text
TEXT
SUBSCRIPTIONID::TEXT
----
BASESTOREID
text
--
text
TEXT
BASESTOREID::TEXT
----
MARKETID
text
--
text
TEXT
MARKETID::TEXT
----
FREQUENCY
text
--
text
TEXT
FREQUENCY::TEXT
----
NEXTORDEREXECUTIONDATE
datetime
--
datetime
TIMESTAMP_NTZ
TRY_TO_TIMESTAMP_NTZ(NEXTORDEREXECUTIONDATE)::TIMESTAMP_NTZ
----
CREATIONDATE
datetime
--
datetime
TIMESTAMP_NTZ
TRY_TO_TIMESTAMP_NTZ(CREATIONDATE)::TIMESTAMP_NTZ
----
PREVIOUSORDEREXECUTIONDATE
datetime
--
datetime
TIMESTAMP_NTZ
TRY_TO_TIMESTAMP_NTZ(PREVIOUSORDEREXECUTIONDATE)::TIMESTAMP_NTZ
----
CANCELLATIONDATE
datetime
--
datetime
TIMESTAMP_NTZ
TRY_TO_TIMESTAMP_NTZ(CANCELLATIONDATE)::TIMESTAMP_NTZ
----
STARTDATE
datetime
--
datetime
TIMESTAMP_NTZ
TRY_TO_TIMESTAMP_NTZ(STARTDATE)::TIMESTAMP_NTZ
----
STATUS
text
--
text
TEXT
STATUS::TEXT
----
UPDATEDAT
datetime
--
datetime
TIMESTAMP_NTZ
TRY_TO_TIMESTAMP_NTZ(UPDATEDAT)::TIMESTAMP_NTZ
----
TEMPLATEBASESTOREID
text
--
text
TEX

Generator with filters

In [10]:
table_header_df = pd.read_excel('table_header - interactions.xlsx')

In [17]:
import json
import pandas as pd
import math
import os
# Replace 'data.json' with the path to your JSON file
# file_path = 'matillion_full.json'
filter_file_path = 'matillion_full_filter.json'
# file_path = 'matillion.json'

# Open and load the JSON file
with open(filter_file_path, 'r', encoding='utf-8') as file:
    data = json.loads(file.read())
# Open and load the JSON file
with open(filter_file_path, 'r', encoding='utf-8') as filter_file:
    filter_data = json.loads(filter_file.read())

# path_header_excel = 'C:\Users\yyusuf5\OneDrive - Philip Morris International\Documents\PMI_CONS\matillion_generator_nb\table_header - interactions.xlsx'

table_header_df = pd.read_excel('table_header - interactions.xlsx')
# table_header_df = pd.read_excel('table_header 2.xlsx')
table_detail_raw_original_df = pd.read_excel('table_detail - interactions.xlsx')
table_detail_raw_df = table_detail_raw_original_df[(table_detail_raw_original_df['target_table_name'].notna())]

def column_priority(col):
    if col.lower() == 'valid_from':
        return 999
    elif col.lower().endswith('_key') and (
            col.lower().startswith('mde_') 
            or col.lower().startswith('tde_')
            or col.lower().startswith('fact_') 
            or col.lower().startswith('dim_')
    ):
        return 0
    elif col.lower().endswith('_bk'):
        return 1
    else:
        return 2


table_detail_raw_df['bk_priority'] = table_detail_raw_df['target_column'].apply(column_priority)


table_detail_df = table_detail_raw_df.sort_values(
    by=['target_table_name', 'primary_key_flag', 'bk_priority', 'target_column'],
    ascending=[True, False, True, True]
).drop(columns='bk_priority') 

data_type_mapping = {
"TEXT(255)": "TEXT(255)",
"TIMESTAMP_NTZ": "TIMESTAMP_NTZ",
"TIMESTAMP_NTZ": "TIMESTAMP_NTZ",
"BOOLEAN": "BOOLEAN",
"NUMBER": "NUMBER",
"text": "TEXT",
"Text": "TEXT",
"Date & Time": "TIMESTAMP_NTZ",
"Decimal (38,4)": "NUMBER(38,5)",
"decimal": "NUMBER(38,5)",
"Boolean": "BOOLEAN",
"flag": "BOOLEAN",
"Integer": "NUMBER",
"integer": "NUMBER",
"Date & Time": "TIMESTAMP_NTZ",
"datetime": "TIMESTAMP_NTZ",
"Date": "DATE"
}

for index,row in table_header_df.iterrows():
    pl_table_lower = row['pl_name'].lower()
    pl_table_upper = row['pl_name'].upper()
    pl_schema_upper = row['pl_schema'].upper()
    staging_schema_upper = row['staging_schema']
    staging_table_upper = row['staging_table'].upper()
    pl_table_description = row['pl_description']
    pl_filter_column = row['filter_column']
    pl_filter_qualifier = row['filter_qualifier']
    pl_filter_operator = row['filter_operator']
    pl_filter_column_value = row['filter_column_value']

    if str(pl_filter_column) != 'nan' and str(pl_filter_qualifier) != 'nan' and str(pl_filter_operator) != 'nan' and str(pl_filter_column_value) != 'nan':
        modified_data = filter_data.copy()
    else:
        modified_data = data.copy()


    table_target_detail_df = pd.DataFrame()
    table_target_detail_df = table_detail_df[table_detail_df['target_table_name'] == pl_table_upper]
    calculator_formula_list = []
    column_description_list = []
    column_classification_list = []
    primary_key_list = []
    md5_key_list = []
    source_column_list = []
    ## prepare calculator data ##
    for index,row in table_target_detail_df.iterrows():
        target_column = row['target_column'] 
        formula = row['column_formula']
        source_column = row['source_column']
        if row['column_data_type'] not in data_type_mapping:
            print(row['column_data_type'])
            break
        else:
            column_data_type = data_type_mapping.get(row['column_data_type'])
        column_description = row['column_description']
        column_classification = row['column_classification']
        column_domain = row['column_domain']
        primary_key_flag = row['primary_key_flag']

        if pd.notna(source_column):
            source_column_list.append(source_column)
        if not pd.notna(formula):
            calculation_formula = f"{source_column}::{column_data_type}"
        else:
            calculation_formula = f"{formula}::{column_data_type}"
        if primary_key_flag == 'Y':
            primary_key_list.append({'values':[target_column]})
            md5_key_list.append([source_column])

        if target_column == 'VALID_FROM':
            if pd.notna(formula):
                md5_key_list.append([formula])
            elif pd.notna(source_column):
                md5_key_list.append([source_column])
            else:
                print(f'{target_column} has issue, no formula or source column defined')


        calculator_formula_list.append({'values':[target_column, calculation_formula]})
        if pd.notna(column_description):
            column_description_list.append({'values':[target_column, column_description.replace("'","\\'")]})


        if pd.notna(column_classification) and pd.notna(column_domain):
            column_classification_list.append({'values':[target_column, column_classification, column_domain]})
        else:
            print(f'{target_column} has issue with column_classification or/and column_domain: column_classification: {column_classification} | column_domain: {column_domain}')
        source_column_string = ','.join(source_column_list)

    # final_calculator_list = [{'values':[f'{pl_table_upper}_KEY','MD5(' + "||'|'||".join([f"UPPER(NVL({col[0]}::varchar,'null'))" for col in md5_key_list]) + "||'|'|| \"_ETL_LOAD_DATETIME\"::varchar)"]}]

    # final_calculator_list.extend(calculator_formula_list)
    gv_calculator_list = []
    for calc in calculator_formula_list:
        gv_calculator_list.append(calc)


    modified_data['transformationJobs'][0]['grids']['gv_calculator']['values'] = gv_calculator_list
    modified_data['orchestrationJobs'][0]['grids']['gv_primary_key_configuration']['values'] = primary_key_list
    modified_data['orchestrationJobs'][0]['grids']['gv_column_description_configuration']['values'] = column_description_list
    modified_data['orchestrationJobs'][0]['grids']['gv_column_classification_configuration']['values'] = column_classification_list


    modified_data_string = json.dumps(modified_data)


    modified_data_string = modified_data_string.replace('$$target_schema$$',pl_schema_upper)
    modified_data_string = modified_data_string.replace('$$target_table_lower$$',pl_table_lower)
    modified_data_string = modified_data_string.replace('$$target_table$$',pl_table_upper)
    modified_data_string = modified_data_string.replace('$$source_schema$$',staging_schema_upper)
    modified_data_string = modified_data_string.replace('$$source_table$$',staging_table_upper)
    modified_data_string = modified_data_string.replace('$$column_list$$',source_column_string)
    modified_data_string = modified_data_string.replace('$$table_description$$',pl_table_description.replace("'","\\\\'"))
    if str(pl_filter_column) != 'nan' and str(pl_filter_qualifier) != 'nan' and str(pl_filter_operator) != 'nan' and str(pl_filter_column_value) != 'nan':
        modified_data_string = modified_data_string.replace('$$filter_column$$',pl_filter_column)
        modified_data_string = modified_data_string.replace('$$filter_qualifier$$',pl_filter_qualifier)
        modified_data_string = modified_data_string.replace('$$filter_operator$$',pl_filter_operator)
        modified_data_string = modified_data_string.replace('$$filter_column_value$$',pl_filter_column_value)

    # # print(modified_data_string)
    # old logic
    # with open(f'matillion_json/{pl_table_lower}.json', 'w') as jsonfile:
    #         print(f'matillion_json/{pl_table_lower}.json')
    #         jsonfile.write(modified_data_string)

    # new logic    
    with open(
        f'matillion_json/{pl_table_lower}.json', 'w', encoding='utf-8') as jsonfile:
        jsonfile.write(modified_data_string)

 